# Data center telemetry anomaly detection

Deep Learning project, Section B. Rishabh Srivastava, Mitul Seth and Madhav Mittal.

This notebook examines the completed experiments and can reproduce training. Normal-only LSTM reconstruction is evaluated against general Nagios server-health labels. The original TensorFlow notebook remains separately available.

Install `requirements.txt` in your environment before running. Keep this notebook beside the `src`, `scripts` and `results` folders. Download the public `0.tar` rack archive into the project root when retraining.

In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/dccad').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from the project folder')
sys.path.insert(0, str(ROOT/'src'))
RETRAIN = False
print('Project:', ROOT.name)
print('RETRAIN:', RETRAIN)

Project: DCCAD
RETRAIN: False


## Data and experimental design

Nine nodes use a chronological 70/10/20 split. Scalers fit finite normal training rows. Causal filling is limited to four readings and resets at gaps. Windows contain known labels, finite readings and uninterrupted 15-minute timestamps. Base experiments use 22 features; the enhanced set uses 33. Training stride is four, while validation and test stride are two.

The delivered runs compare 8, 24 and 48 readings, enhanced features and robust scaling. Each LSTM uses hidden size 64, latent size 16, seed 42, Adam 0.001, batch 512 and at most 30 epochs. Setting selection uses labelled validation F1, rather than test performance.

In [2]:
evidence = json.loads((ROOT/'results/submission_evidence.json').read_text())
print('Nodes:', evidence['dataset']['nodes'])
print('Reduced readings:', evidence['dataset']['rows'])
print('Label counts:', evidence['dataset']['labels'])
print('Split row counts:', evidence['dataset']['split_rows'])

Nodes: [7, 8, 9, 10, 11, 13, 14, 16, 17]
Reduced readings: 751363
Label counts: {'-1': 81715, '0': 660172, '2': 9429, '3': 47}
Split row counts: {'train': 525950, 'validation': 75137, 'test': 150276}


## Optional full reproduction

Set `RETRAIN = True` in the first cell to rerun the five neural configurations and the Isolation Forest baseline. Training takes substantial CPU time. The default uses the trained artifacts already supplied. The raw archive is needed for full reproduction.

In [3]:
if RETRAIN:
    if not (ROOT/'0.tar').exists():
        raise FileNotFoundError('Download 0.tar from https://zenodo.org/records/7541722')
    for script, args in [('evaluate_saved.py', []), ('train_baseline.py', []), ('train_experiments.py', ['--epochs','30','--configs','base24','base8','base48','enhanced24','robust24']), ('build_results.py', [])]:
        subprocess.run([sys.executable, str(ROOT/'scripts'/script), *args], cwd=ROOT, check=True)
else:
    print('Using completed experiment artifacts')

Using completed experiment artifacts


## Correct endpoint and incident evaluation

`y_window` marks any CRITICAL reading in a window. `y_end` uses only the original label at the scored timestamp. Raw incidents group consecutive CRITICAL 15-minute bins, with breaks at missing timestamps. Alarm timestamps must fall inside the original interval to count. Both total and observable incident recall are retained. Persistence and alarm bursts reset at gaps and node changes. AP refers to average precision, not trapezoidal PR-AUC.

In [4]:
audit = json.loads((ROOT/'results/original_audit/summary.json').read_text())
print(json.dumps(audit, indent=2))
comparison = pd.read_csv(ROOT/'results/experiment_comparison.csv')
print(comparison[['experiment','width','validation_f1','average_precision','precision','recall','f1','detected','episodes','observable_episodes','false_bursts_per_node_day']].to_string(index=False))

{
  "test_windows": 72252,
  "window_positives": 761,
  "endpoint_positives": 432,
  "raw_incidents": 34,
  "alignment_verified": true,
  "status": "Fixed policy sensitivity analysis on previously inspected test scores; not validation-selected improvements"
}
        experiment  width  validation_f1  average_precision  precision   recall       f1  detected  episodes  observable_episodes  false_bursts_per_node_day
            base24     24       0.027460           0.096220   0.019254 0.708333 0.037489        16        34                   22                   0.451088
            base48     48       0.014363           0.145048   0.053180 0.616967 0.097919        11        34                   22                   0.486158
             base8      8       0.074968           0.103086   0.021134 0.722656 0.041068        17        34                   22                   0.691514
         clean22_8      8       0.074362           0.101794   0.020848 0.691406 0.040476        17        34    

## Selected LSTM and limitations

The validation-selected LSTM is **cooling12_8**. Its endpoint average precision is **0.0789** and its selected-policy F1 is **0.0725** on the corresponding test cohort. It catches **15 of 34 raw incidents**; **22** have scoring opportunities. Different window lengths change retained cohorts. General server-health labels, a single rack, one seed, overlapping windows and previously inspected test recordings limit interpretation. The baseline can outperform the LSTM on ranking metrics.

In [5]:
selected = json.loads((ROOT/'results/selected_model.json').read_text())['name']
summary = json.loads((ROOT/'results'/selected/'summary.json').read_text())
print('Selected policy:', json.dumps({k:summary['selected_policy'][k] for k in ['score_mode','quantile','per_node','persistence']}, indent=2))
print('Per-node results:')
print(pd.read_csv(ROOT/'results'/selected/'per_node.csv').to_string(index=False))

Selected policy: {
  "score_mode": "whole_window",
  "quantile": 95.0,
  "per_node": true,
  "persistence": 1
}
Per-node results:
 node  samples  positives  roc_auc  average_precision  precision   recall       f1  tp   fp  fn   tn
    7     8247         16 0.826487           0.043424   0.012225 0.625000 0.023981  10  808   6 7423
    8     8071         15 0.822435           0.033644   0.012009 0.733333 0.023631  11  905   4 7151
    9     8246         15 0.781647           0.371314   0.005362 0.666667 0.010638  10 1855   5 6376
   10     8237        397 0.836958           0.238901   0.191691 0.662469 0.297343 263 1109 134 6731
   11     8247         13 0.945535           0.108487   0.013873 0.923077 0.027335  12  853   1 7381
   13     8072         13 0.937700           0.164855   0.015645 0.923077 0.030769  12  755   1 7304
   14     8247         13 0.887848           0.160954   0.009913 0.615385 0.019512   8  799   5 7435
   16     7861         17 0.889957           0.077823   0.0128

## Fresh saved-model inference

The example file contains actual consecutive 15-minute feature readings. This cell loads the selected checkpoint and scaler, calculates a new score and lists the largest latest-reading reconstruction errors. An isolated window does not establish a persistence streak.

In [6]:
sys.path.insert(0,str(ROOT/'scripts'))
from predict import predict
example = json.loads((ROOT/'results'/selected/'example_metadata.json').read_text())
print(json.dumps(predict(ROOT/'results'/selected, ROOT/'results'/selected/'example_window.csv', example['node']), indent=2))

{
  "score": 0.0010208802996203303,
  "threshold": 0.006795273162424564,
  "above_threshold": false,
  "persistence_required": 1,
  "top_latest_error_sensors": [
    "p0_core_temp",
    "p0_vdd_temp",
    "fan1_0"
  ]
}


## Data cleaning and feature selection

The supplementary workflow preserves the original data, flags nonfinite and implausible negative sensor measurements, and forward-fills at most four readings without crossing timestamp gaps or split boundaries. Selection uses normal training data only: missingness, constant inputs and redundant correlations. Correlation selection and a predefined cooling subset are compared with cleaned and original 22-feature models on exactly the same window endpoints. These experiments remain exploratory. The earlier final PDF and PPT are supplemented by `results/feature_selection/REPORT.md`.

In [7]:
feature_dir = ROOT/'results/feature_selection'
if (feature_dir/'results.json').exists():
    feature_result = json.loads((feature_dir/'results.json').read_text())
    print('Selected input count:', feature_result['plan']['feature_count'])
    print('Matched-cohort validation winner:', feature_result['winner_on_matched_cohort'])
    print(pd.read_csv(feature_dir/'comparison.csv')[['experiment','features','validation_f1','precision','recall','f1','average_precision']].to_string(index=False))
else:
    print('Run scripts/clean_and_select.py to generate supplementary results')

Selected input count: 16
Matched-cohort validation winner: cooling12_8
        experiment  features  validation_f1  precision   recall       f1  average_precision
feature_reference8        22       0.074968   0.021134 0.722656 0.041068           0.103086
         clean22_8        22       0.074362   0.020848 0.691406 0.040476           0.101794
      correlation8        16       0.074839   0.031415 0.685547 0.060077           0.123093
       cooling12_8        12       0.076428   0.038271 0.679688 0.072462           0.078917


## Dashboard and submission

Run `python -m streamlit run dashboard/app.py` from the project folder. The dashboard includes recorded replay, sensor reconstructions, experiment tables and fresh CSV scoring.

The report and final PPT are in `submission`. Detailed histories, scores and incident records are in `results`. The README contains commands, feature definitions, data sources and scope limitations.